# V2 Model Reliability & Controlled Evaluation — Experimental Demonstration

**PRELIMINARY / EXPERIMENTAL.** A short mentor walkthrough of the current V2 dependency-classification pipeline and its execution controls.

**Problem → desired behavior → current approach → experiments → evidence → limitations.**

Three evidence types are kept separate: **recorded historical response**, **live V2 utility executions with synthetic fixtures**, and **optional real checkpoint replay**. Synthetic probabilities are never presented as model predictions.

Source reference: `Its-soul/cap-stone`, GitHub revision `0548f0cf1bff98a518781d706c9780d290728cab`, with the existing V2 test setup. No production code or V3 changes are required.

*Output provenance: cells executed with in-process IPython because this environment disallows Jupyter kernel socket binding. The real-model section was NOT RUN: no private V2 bundle was configured.*

## Problem

The project tracks which evidence supports an artifact, invalidates affected certificates when evidence changes, and preserves independent valid work during recovery. The deterministic certificate engine remains the authority. V2 supplies **advisory binary dependency predictions**.

Three practical risks matter: a confident semantic prediction can be wrong; changed data or weights can make comparisons invalid; and out-of-order or failed execution can leave misleading results. V2 addresses the experiment-control risks and tests a broader training distribution. Its predictive improvement must be measured separately.

**Important target limitation:** the reviewed target is declared transitive support membership for a named artifact. V2's synthetic training protocol describes closed-function sensitivity. These are not equivalent for zero-effect reads, redundant evidence, or cancellation. This notebook does not claim V2 has resolved that gap.

### Setup

Open this notebook from the repository root, or set `CERT_RECOVERY_REPO` to your checkout. Use a Jupyter kernel with the project's existing test dependencies (`pytest`, `PyYAML`). Real replay additionally needs the **existing private model environment**, its local V2 bundle, and `CERT_RECOVERY_V2_RUN`.

This notebook performs no training, downloads, remote requests, or V3 execution. Fixture writes go only to a disposable temporary directory. Private replay reads validation examples and preserves the frozen threshold; it does not select a new model or evaluate a new final set.

In [1]:
import os, sys, json, html, tempfile, importlib.util, math
from pathlib import Path
from types import SimpleNamespace
from unittest.mock import Mock, patch
from IPython.display import display, HTML, Markdown

REPO_LOCATION = os.environ.get("CERT_RECOVERY_REPO", "")
V2_LOCATION = os.environ.get("CERT_RECOVERY_V2_RUN", "")
candidates = [Path(REPO_LOCATION).expanduser()] if REPO_LOCATION else [
    Path.cwd(), *Path.cwd().parents, Path.cwd() / "cap-stone-current"]
ROOT = next((p.resolve() for p in candidates
             if (p / "src/cert_recovery/controlled_study.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Set CERT_RECOVERY_REPO to your current repository checkout, then rerun.")
sys.path.insert(0, str(ROOT / "src"))
from cert_recovery.data_pipeline import clean_rows, load_verified_splits, write_json
from cert_recovery.model_pipeline import _infer, _prediction_report
from cert_recovery.controlled_study import verify_frozen, execute_stage
from cert_recovery.pinned_smoke import sha256_file
import cert_recovery.controlled_study as study

def table(rows, columns=None):
    columns = columns or list(rows[0]) if rows else columns or []
    def escaped(value):
        if isinstance(value, (dict, list)):
            value = json.dumps(value, ensure_ascii=False)
        return html.escape(str(value)).replace("\n", "<br>")
    body = ''.join('<tr>' + ''.join('<td>' + escaped(r.get(c, '')) + '</td>' for c in columns)
                   + '</tr>' for r in rows)
    display(HTML('<style>table.demo{border-collapse:collapse;font:14px system-ui;max-width:100%}'
                 'table.demo th,table.demo td{border:1px solid #ccd4de;padding:9px;text-align:left;'
                 'vertical-align:top;overflow-wrap:anywhere}table.demo th{background:#edf2f8}</style>'
                 '<table class="demo"><tr>' + ''.join('<th>' + escaped(c) + '</th>' for c in columns)
                 + '</tr>' + body + '</table>'))

def observed_error(action):
    try:
        action()
    except Exception as error:
        return {"type": type(error).__name__, "message": str(error)}
    return {"type": "NO_ERROR", "message": "No rejection occurred"}

evidence, before_after = [], []
table([{"Source": "Verified current checkout + existing V2 test fixtures",
        "Training": "Not executed", "Model replay": "Local bundle opt-in only",
        "V3": "Not imported or executed"}])

Source,Training,Model replay,V3
Verified current checkout + existing V2 test fixtures,Not executed,Local bundle opt-in only,Not imported or executed


## Existing Behavior

**Input:** “A man is eating pizza.” / “The man owns a bicycle.”

**Expected:** NLI **neutral**; eating pizza does not determine bicycle ownership.

**Actual below:** load the genuine recorded Space response from the repository fixture. This is historical evidence, not fresh inference. The historical service's loaded weights/revision were unknown.

**Why it matters:** successful transport and high reported confidence do not establish semantic correctness. NLI is a different task from binary support membership; this example is **not** an original-vs-V2 dependency comparison.

In [2]:
fixture = ROOT / "tests/remote/fixtures/case_003_recorded_response.json"
historical = json.loads(fixture.read_text(encoding="utf-8"))
record = historical["record"]
normal = record["normalized_output"]
table([{"Evidence": historical["fixture_type"],
        "Input": record["input"], "Expected NLI": record["expected_label"],
        "Actual NLI": normal["label"], "Reported confidence": normal["confidence_text"],
        "Matches expectation": record["matches_expected_label"],
        "Loaded model identity": record["model_identity_status"]}])
assert record["expected_label"] == "neutral" and normal["label"] == "contradiction"
evidence.append({"Area": "Historical semantic failure", "Experiment": "Read recorded case 003",
                 "Actual observation": "Neutral annotation; contradiction at 99.57%; no fresh inference"})
display(Markdown("**Observation:** incorrect, high-confidence historical NLI response. "
                 "The original binary checkpoint is not available here; its historical predictions "
                 "cannot be independently reproduced in this environment."))

Evidence,Input,Expected NLI,Actual NLI,Reported confidence,Matches expectation,Loaded model identity
recorded_response_not_new_inference,"{""premise"": ""A man is eating pizza."", ""hypothesis"": ""The man owns a bicycle.""}",neutral,contradiction,99.57%,False,UNKNOWN


**Observation:** incorrect, high-confidence historical NLI response. The original binary checkpoint is not available here; its historical predictions cannot be independently reproduced in this environment.

## V2 Goal

**EXPECTED / THEORETICAL — goals, not measured improvements.**

| Scenario | Problematic behavior | Desired V2 behavior |
|---|---|---|
| Complete, scoped input | Lose negation, values, or the named target | Preserve the complete model-visible pair |
| Dependency decision | Treat confidence or NLI as proof | Use the frozen binary threshold; keep advice separate from certificate validity |
| Controlled comparison | Change data, source, or selection between runs | Reject artifacts that differ from frozen hashes |
| Stage execution | Run comparisons too early or overwrite failure evidence | Enforce prerequisites and retain each attempt's status |
| Structural reasoning | Learn identifiers or phrasing instead of the declared support relation | Generalize under controlled role changes; this requires further evidence |

## V2 Approach

**Frozen protocol + data + source hashes** → **verified splits** → **complete premise/hypothesis tokenization** → **DeBERTa binary head** → **softmax probability + frozen threshold** → **prediction report + saved comparison**.

The stage controller wraps execution: **training → proxy selection → comparison**, with prerequisite checks and preserved attempt records.

- **Cleaning / splits:** NFC Unicode and whitespace normalization; reject malformed or conflicting rows and split leakage.
- **Model:** `cross-encoder/nli-deberta-v3-small`, with a fresh two-class dependency head in V2 training. Exact revision, limits and runtime choices come from the private frozen run, not the public default configuration.
- **Selection:** validation chooses the checkpoint and threshold; final predictions do not choose them.
- **Reporting:** `_infer` captures logits and all class probabilities; `_prediction_report` applies `p >= threshold`, records correctness and class support.
- **Integrity / authority:** `verify_frozen` checks the run; `execute_stage` controls attempts. Model advice cannot replace the independent deterministic verifier.

We reuse these functions and the existing `frozen_run` fixture from `tests/test_v2_execution.py`. The fixture contains six explicitly synthetic rows and real source hashes, **no checkpoint or trained model**.

In [3]:
fixture_spec = importlib.util.spec_from_file_location(
    "mentor_v2_existing_fixture", ROOT / "tests/test_v2_execution.py")
fixture_module = importlib.util.module_from_spec(fixture_spec)
fixture_spec.loader.exec_module(fixture_module)
scratch = tempfile.TemporaryDirectory(prefix="v2_mentor_demo_")
scratch_root = Path(scratch.name)
def fresh_fixture(name):
    folder = scratch_root / name
    folder.mkdir()
    return fixture_module.frozen_run.__wrapped__(folder)
table([{"Demonstration backend": "Existing frozen_run fixture",
        "Rows": "6 synthetic rows; separate train/validation/test groups",
        "Revision field": "Synthetic placeholder; never loaded",
        "Writes": "Disposable temporary directory only"}])

Demonstration backend,Rows,Revision field,Writes
Existing frozen_run fixture,6 synthetic rows; separate train/validation/test groups,Synthetic placeholder; never loaded,Disposable temporary directory only


## Test Cases

### Test Case 1 — Preserve a complete scoped input

**Input:** a synthetic pair with decomposed Unicode, repeated whitespace, an explicit Q-only scope, negation, and the value 17.

**Before / problem:** the raw string has inconsistent formatting. Losing the scope or “not” during cleanup would change the task.

**Expected:** canonical Unicode and single spaces; preserve `Q-only`, `not`, `17`, and `café`. This tests text handling, not a learned dependency prediction.

**Run V2 / Actual Output:**

In [4]:
raw_pair = {"pair_id": "mentor-clean", "world_id": "mentor-world",
            "premise": "  Q-only:\n source Cafe\u0301 is not 17.  All sources have no supports. ",
            "hypothesis": " Q uses Cafe\u0301 only.\tAll other sources are excluded. ",
            "label": 1, "source": "Explicitly synthetic text-handling demonstration"}
cleaned, cleaning = clean_rows([raw_pair])
accepted = cleaned[0]
expected_text = "Q-only: source Café is not 17. All sources have no supports."
assert accepted["premise"] == expected_text
assert accepted["hypothesis"] == "Q uses Café only. All other sources are excluded."
table([{"Field": field, "Before": repr(raw_pair[field]), "Expected": expected,
        "Actual": accepted[field], "Match": accepted[field] == expected}
       for field, expected in [("premise", expected_text),
                               ("hypothesis", "Q uses Café only. All other sources are excluded.")]])
evidence.append({"Area": "Input handling", "Experiment": "Real clean_rows on a synthetic pair",
                 "Actual observation": "Whitespace/NFC cleaned; scope, negation and 17 preserved"})
before_after.append({"Test case": "Complete scoped input", "Before": "Raw inconsistent whitespace/Unicode",
                     "V2": "Exact expected normalized text", "Difference": "Formatting only; task text preserved"})
display(Markdown("**Observation:** correct text handling for this example. No model accuracy claim."))

Field,Before,Expected,Actual,Match
premise,' Q-only:\n source Café is not 17. All sources have no supports. ',Q-only: source Café is not 17. All sources have no supports.,Q-only: source Café is not 17. All sources have no supports.,True
hypothesis,' Q uses Café only.\tAll other sources are excluded. ',Q uses Café only. All other sources are excluded.,Q uses Café only. All other sources are excluded.,True


**Observation:** correct text handling for this example. No model accuracy claim.

### Test Case 2 — A positive prediction can have probability below 0.5

**Input:** an explicitly **synthetic** dependency probability of 0.30, with illustrative frozen threshold 0.20 and provisional label 1.

**Before / problem:** applying a conventional 0.50 cutoff gives class 0. It is not the V2 frozen decision rule in this illustration.

**Expected:** V2 predicts class 1 because `0.30 >= 0.20`. Its reported predicted-label probability remains 0.30; this is not a high-confidence prediction.

**Run V2 / Actual Output:** the real reporting function; **no model forward pass and no historical baseline**.

In [5]:
synthetic_probability = 0.30
illustrative_threshold = 0.20
decision_report = _prediction_report([accepted], [synthetic_probability], illustrative_threshold, 10)
decision = decision_report["predictions"][0]
assert decision["prediction"] == 1 and decision["predicted_label_probability"] == 0.30
table([{"Input origin": "Synthetic probability; NOT model output",
        "P(depends_on)": decision["probability"], "Threshold": decision_report["threshold"],
        "Expected / actual prediction": f"1 / {decision['prediction']}",
        "All binary probabilities": decision["class_probabilities"],
        "Predicted-label probability": decision["predicted_label_probability"],
        "Illustrative 0.50 decision": int(synthetic_probability >= 0.50)}])
bad_probability = observed_error(lambda: _prediction_report([accepted], [float('nan')], 0.20, 10))
assert bad_probability["type"] == "ValueError"
table([{"Malformed input": "NaN probability", "Actual rejection": bad_probability}])
evidence.append({"Area": "Threshold behavior", "Experiment": "Real reporter; injected probability 0.30",
                 "Actual observation": "Threshold .20 yields label 1 with probability .30; NaN rejected"})
before_after.append({"Test case": "Threshold decision (synthetic)", "Before": "Illustrative .50 cutoff: label 0",
                     "V2": "Illustrative frozen .20 cutoff: label 1",
                     "Difference": "Shows the actual decision rule; not a measured model improvement"})
display(Markdown("**Observation:** decision logic behaves as expected. A lower threshold trades "
                 "missed dependencies against false alarms; this example does not establish the best threshold."))

Input origin,P(depends_on),Threshold,Expected / actual prediction,All binary probabilities,Predicted-label probability,Illustrative 0.50 decision
Synthetic probability; NOT model output,0.3,0.2,1 / 1,"[0.7, 0.3]",0.3,0


Malformed input,Actual rejection
NaN probability,"{""type"": ""ValueError"", ""message"": ""Probabilities must be finite and in [0,1]""}"


**Observation:** decision logic behaves as expected. A lower threshold trades missed dependencies against false alarms; this example does not establish the best threshold.

### Test Case 3 — Reject changed validation evidence

**Input:** a valid bundle from the existing synthetic fixture, followed by a changed validation premise in its disposable copy.

**Before / problem:** the input would differ from the frozen dataset. Silently evaluating it would compromise comparability.

**Expected:** accept the untouched fixture; reject the altered split. No historical implementation is run or alleged to lack this protection.

**Run V2 / Actual Output:**

In [6]:
integrity_run = fresh_fixture("integrity")
verified_config, verified_protocol = verify_frozen(integrity_run)
validation_path = integrity_run / "splits/validation.jsonl"
untouched_bytes = validation_path.read_bytes()
validation_rows = [json.loads(line) for line in validation_path.read_text().splitlines()]
validation_rows[0]["premise"] = "Altered demonstration text"
validation_path.write_text(''.join(json.dumps(r) + "\n" for r in validation_rows))
integrity_error = observed_error(lambda: verify_frozen(integrity_run))
assert integrity_error == {"type": "ValueError", "message": "Empty or altered split: validation"}
validation_path.write_bytes(untouched_bytes)
verify_frozen(integrity_run)
table([{"Bundle state": "Untouched", "Expected": "Accept", "Actual": "Accepted by verify_frozen"},
       {"Bundle state": "Validation text changed", "Expected": "Reject", "Actual": integrity_error},
       {"Bundle state": "Original bytes restored", "Expected": "Accept", "Actual": "Accepted again"}])
evidence.append({"Area": "Artifact integrity", "Experiment": "Alter disposable validation text",
                 "Actual observation": "ValueError: Empty or altered split: validation"})
before_after.append({"Test case": "Artifact integrity", "Before": "Valid frozen copy accepted",
                     "V2": "Changed copy rejected", "Difference": "Change detected; restored bytes accepted"})
display(Markdown("**Observation:** the integrity check catches this change. "
                 "It does not prove the annotations themselves are correct."))

Bundle state,Expected,Actual
Untouched,Accept,Accepted by verify_frozen
Validation text changed,Reject,"{""type"": ""ValueError"", ""message"": ""Empty or altered split: validation""}"
Original bytes restored,Accept,Accepted again


**Observation:** the integrity check catches this change. It does not prove the annotations themselves are correct.

## V2 Reliability Demonstrations

### Stage ordering

**Input / problem:** request comparison before training or proxy selection.

**Expected:** V2 rejects it before creating an attempt. **Run V2 / Actual Output:**

In [7]:
ordering_run = fresh_fixture("ordering")
manifest_before = (ordering_run / "manifest.json").read_bytes()
ordering_error = observed_error(lambda: execute_stage(ordering_run, "comparison"))
attempts = len(list(ordering_run.glob("attempt_*")))
assert ordering_error["message"] == "Successful selected training required"
assert attempts == 0 and (ordering_run / "manifest.json").read_bytes() == manifest_before
table([{"Input": "comparison with no successful training", "Expected": "Reject before attempt",
        "Actual": ordering_error, "Attempts created": attempts, "Manifest changed": False}])
evidence.append({"Area": "Execution control", "Experiment": "Comparison before training",
                 "Actual observation": "Rejected; zero attempts; manifest unchanged"})
display(Markdown("**Observation:** the invalid order was blocked without running a model."))

Input,Expected,Actual,Attempts created,Manifest changed
comparison with no successful training,Reject before attempt,"{""type"": ""ValueError"", ""message"": ""Successful selected training required""}",0,False


**Observation:** the invalid order was blocked without running a model.

### Failure preservation and CPU-only enforcement

**Controlled fault injection — not real training.** Reuse the existing test strategy: substitute only the external training callback and a tiny hardware adapter, then execute the **real V2 stage controller** on a disposable fixture.

**Input / problem:** a training backend raises an error, or the simulated hardware reports a GPU despite this protocol being CPU-only.

**Expected:** a persisted `ERROR` record in both the attempt and run manifest. A second attempt on the same failed stage must be rejected. These notebook-local substitutions are restored immediately.

In [8]:
failure_run = fresh_fixture("failure")
hardware_adapter = SimpleNamespace(set_num_threads=Mock(), cuda=SimpleNamespace(is_available=lambda: False))
with patch.dict(sys.modules, {"torch": hardware_adapter}), patch.object(
        study, "run_finetuning", side_effect=RuntimeError("Controlled mentor-demo backend failure")) as backend:
    failure_error = observed_error(lambda: execute_stage(failure_run, "training"))
    calls = backend.call_count
failure_manifest = json.loads((failure_run / "manifest.json").read_text())
failure_state = failure_manifest["stages"]["training"]
persisted = json.loads((Path(failure_state["attempt"]) / "status.json").read_text())
assert persisted == failure_state and failure_state["status"] == "ERROR" and calls == 1
retry_error = observed_error(lambda: execute_stage(failure_run, "training"))
assert retry_error["type"] == "FileExistsError"
table([{"Event": "Injected backend exception", "Expected": "ERROR evidence retained",
        "Actual": failure_error, "Persisted status": persisted["status"],
        "Attempt/manifest agree": persisted == failure_state},
       {"Event": "Retry same stage", "Expected": "Reject overwrite/retry",
        "Actual": retry_error, "Persisted status": failure_state["status"],
        "Attempt/manifest agree": True}])
gpu_run = fresh_fixture("simulated_gpu")
gpu_adapter = SimpleNamespace(set_num_threads=Mock(), cuda=SimpleNamespace(is_available=lambda: True))
with patch.dict(sys.modules, {"torch": gpu_adapter}), patch.object(study, "run_finetuning") as forbidden_backend:
    gpu_error = observed_error(lambda: execute_stage(gpu_run, "training"))
    backend_called = forbidden_backend.called
gpu_state = json.loads((gpu_run / "manifest.json").read_text())["stages"]["training"]
assert gpu_error["message"] == "This frozen experiment is local CPU only" and not backend_called
assert gpu_state["status"] == "ERROR"
table([{"Simulation": "GPU available", "Expected": "Block this CPU-only protocol",
        "Actual": gpu_error, "Training callback called": backend_called,
        "Persisted status": gpu_state["status"]}])
evidence.extend([
    {"Area": "Failure handling", "Experiment": "Synthetic backend exception + retry",
     "Actual observation": "ERROR retained in both records; retry rejected"},
    {"Area": "CPU-only guard", "Experiment": "Simulated GPU availability",
     "Actual observation": "Rejected and recorded ERROR; training callback not called"}])
display(Markdown("**Observation:** failures remain visible. This protocol-specific CPU guard "
                 "does not imply that DeBERTa generally cannot run on a GPU."))

Event,Expected,Actual,Persisted status,Attempt/manifest agree
Injected backend exception,ERROR evidence retained,"{""type"": ""RuntimeError"", ""message"": ""Controlled mentor-demo backend failure""}",ERROR,True
Retry same stage,Reject overwrite/retry,"{""type"": ""FileExistsError"", ""message"": ""Stage already attempted; preserve attempt and choose a separately authorized new run""}",ERROR,True


Simulation,Expected,Actual,Training callback called,Persisted status
GPU available,Block this CPU-only protocol,"{""type"": ""ValueError"", ""message"": ""This frozen experiment is local CPU only""}",False,ERROR


**Observation:** failures remain visible. This protocol-specific CPU guard does not imply that DeBERTa generally cannot run on a GPU.

## Real V2 Checkpoint Replay

**Configure:** set `CERT_RECOVERY_V2_RUN` to your existing private V2 run folder before starting this notebook, and select its existing model-environment kernel. Rerun all cells.

We follow `tests/integration/test_v2_checkpoint.py`: `verify_frozen` → checkpoint file hashes → local-only tokenizer/model loading → verified validation rows → `_infer` → `_prediction_report` → saved-prediction comparison. No checkpoint is created and no threshold is retuned.

Choose up to three **validation** rows from the saved report: one correct dependent case, one correct independent case, and the highest-confidence error if present. This is a deliberately selected qualitative demonstration, **not an unbiased accuracy estimate**. Annotation labels are provisional expectations; agreement with a saved prediction checks reproducibility, not truth.

If the private bundle is missing, the section displays **NOT RUN** and the notebook remains executable. An available but invalid bundle displays **ERROR**; it is not bypassed or replaced with synthetic inference.

In [9]:
replay_status, replay_results, replay_report = "NOT RUN", [], None
replay_error, actual_config, actual_selection = None, None, None
if not V2_LOCATION:
    replay_reason = "Private V2 bundle not configured; no model outputs generated."
else:
    try:
        run = Path(V2_LOCATION).expanduser().resolve()
        actual_config, protocol = verify_frozen(run)
        actual_selection = json.loads((run / "checkpoints/selection.json").read_text())
        checkpoint = run / "checkpoints/best"
        if sha256_file(checkpoint / "model.safetensors") != actual_selection["checkpoint_sha256"]:
            raise ValueError("V2 checkpoint hash mismatch")
        for filename, digest in actual_selection["checkpoint_file_hashes"].items():
            if sha256_file(checkpoint / filename) != digest:
                raise ValueError(f"Checkpoint file changed: {filename}")
        splits, _ = load_verified_splits(actual_config)
        saved = json.loads((run / "comparisons/validation_candidate_v2.json").read_text())
        saved_by_id = {r["pair_id"]: r for r in saved["predictions"]}
        for row in splits["validation"]:
            recorded = saved_by_id[row["pair_id"]]
            if any(recorded[k] != row[k] for k in ("premise", "hypothesis", "label")):
                raise ValueError("Saved prediction input/label differs from verified validation row")
        selected_ids = []
        for label in (1, 0):
            eligible = [r for r in saved["predictions"] if r["label"] == label and r["prediction"] == label]
            if eligible:
                selected_ids.append(eligible[0]["pair_id"])
        incorrect = [r for r in saved["predictions"] if r["prediction"] != r["label"]]
        if incorrect:
            selected_ids.append(max(incorrect, key=lambda r: r["probability"] if r["prediction"] else 1-r["probability"])["pair_id"])
        for row in splits["validation"]:
            if len(selected_ids) >= 3:
                break
            if row["pair_id"] not in selected_ids:
                selected_ids.append(row["pair_id"])
        verified_by_id = {r["pair_id"]: r for r in splits["validation"]}
        replay_rows = [verified_by_id[p] for p in selected_ids]
        os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        tokenizer = AutoTokenizer.from_pretrained(checkpoint, local_files_only=True, trust_remote_code=False)
        model = AutoModelForSequenceClassification.from_pretrained(
            checkpoint, local_files_only=True, trust_remote_code=False, use_safetensors=True)
        if model.config.id2label != {0: "independent", 1: "depends_on"}:
            raise ValueError("Unexpected V2 binary label mapping")
        for row in replay_rows:
            if len(tokenizer(row["premise"], row["hypothesis"], truncation=False)["input_ids"]) > actual_config["model"]["max_length"]:
                raise ValueError("Replay would truncate complete task context")
        capture = []
        with patch.object(torch.cuda, "is_available", return_value=False):
            probabilities = _infer(actual_config, replay_rows, model, tokenizer, 1, capture)
        replay_report = _prediction_report(replay_rows, probabilities, actual_selection["threshold"],
                                            actual_config["model"]["calibration_bins"], capture)
        for actual in replay_report["predictions"]:
            recorded = saved_by_id[actual["pair_id"]]
            replay_results.append({"Pair ID": actual["pair_id"],
                "Annotation expected": actual["label"], "Saved V2 prediction": recorded["prediction"],
                "Actual V2 prediction": actual["prediction"], "P(depends_on)": actual["probability"],
                "Frozen threshold": actual_selection["threshold"], "Logits": actual["raw_logits"],
                "All probabilities": actual["original_model_class_probabilities"],
                "Matches annotation": actual["prediction"] == actual["label"],
                "Matches saved prediction": actual["prediction"] == recorded["prediction"],
                "Probability replay within 1e-5": math.isclose(actual["probability"], recorded["probability"], abs_tol=1e-5, rel_tol=1e-5)})
        replay_status = "EXECUTED"
        replay_reason = "Read-only local CPU replay of selected validation rows; no selection changes."
        table([{"Model": actual_config["model"]["pretrained_id"],
                "Pinned base revision": actual_config["model"]["revision"],
                "Checkpoint SHA-256": actual_selection["checkpoint_sha256"],
                "Head": model.config.id2label, "Maximum tokens": actual_config["model"]["max_length"],
                "Training configuration (recorded, not rerun)": actual_config["training"]}])
        del model, tokenizer
    except Exception as error:
        replay_status = "ERROR"
        replay_error = {"type": type(error).__name__, "message": str(error)}
        replay_reason = "Replay stopped; repair the local bundle/environment without weakening its guards."
table([{"Replay status": replay_status, "Observation": replay_reason, "Error": replay_error}])
evidence.append({"Area": "Real model inference", "Experiment": "Private validation checkpoint replay",
                 "Actual observation": f"{replay_status}: {replay_reason}"})

Replay status,Observation,Error
NOT RUN,Private V2 bundle not configured; no model outputs generated.,None


In [10]:
if replay_report is None:
    display(Markdown("**Actual V2 model outputs: unavailable in this execution.** "
                     "The populated utility experiments above are not substitute model predictions."))
else:
    for number, (input_row, values) in enumerate(zip(replay_rows, replay_results), 1):
        display(Markdown(f"### Model example {number} — `{input_row['pair_id']}`"))
        display(Markdown("**Input**"))
        table([{"Premise": input_row["premise"], "Hypothesis": input_row["hypothesis"]}])
        display(Markdown("**Expected:** the provisional annotation below; saved V2 output is a separate replay reference."
                         "\n\n**Run V2:** `_infer` and `_prediction_report` executed in the preceding cell."
                         "\n\n**Actual Output**"))
        table([values])
        correct = values["Matches annotation"]
        reproducible = values["Matches saved prediction"] and values["Probability replay within 1e-5"]
        display(Markdown("**Observation:** " + ("matches" if correct else "does not match")
                         + " the provisional annotation; " + ("reproduces" if reproducible else "differs from")
                         + " the saved V2 prediction. Confidence remains advisory."))

**Actual V2 model outputs: unavailable in this execution.** The populated utility experiments above are not substitute model predictions.

## Before vs V2

**Utility experiments:** compare raw/valid/simulated conditions with the actual guarded behavior. The 0.50 rule is illustrative, not a historical model.

**Model comparison:** when available, read the original and V2 saved validation reports for the **same verified inputs**. Saved artifacts are not fresh original inference. Without private artifacts, no numerical original-vs-V2 improvement is claimed.

In [11]:
table(before_after)
model_comparison = []
if replay_report is not None:
    original_file = run / "comparisons/validation_original.json"
    if original_file.is_file():
        original_report = json.loads(original_file.read_text())
        original_by_id = {r["pair_id"]: r for r in original_report["predictions"]}
        for actual in replay_report["predictions"]:
            before = original_by_id.get(actual["pair_id"])
            if before is None or any(before[k] != actual[k] for k in ("premise", "hypothesis", "label")):
                raise ValueError("Original comparator does not match the verified input/label")
            before_correct = before["prediction"] == actual["label"]
            now_correct = actual["prediction"] == actual["label"]
            difference = ("Improvement on provisional label" if not before_correct and now_correct else
                          "Regression on provisional label" if before_correct and not now_correct else
                          "Both match provisional label" if now_correct else "Both disagree with provisional label")
            model_comparison.append({"Pair ID": actual["pair_id"], "Expected annotation": actual["label"],
                "Before (saved original)": before["prediction"], "Before probability": before["probability"],
                "Before threshold": original_report["threshold"], "Before logits": before.get("raw_logits", "Not recorded"),
                "V2 (fresh replay)": actual["prediction"], "V2 probability": actual["probability"],
                "V2 threshold": replay_report["threshold"], "V2 logits": actual["raw_logits"], "Difference": difference})
        table(model_comparison)
    else:
        display(Markdown("**Original comparison NOT AVAILABLE:** saved validation_original.json is missing. "
                         "No historical output is inferred from V2."))
else:
    table([{"Model before/after": "NOT AVAILABLE in this execution",
            "Reason": "Private original/V2 validation artifacts and checkpoint are not available",
            "Claimed predictive improvement": "None from this notebook execution"}])

Test case,Before,V2,Difference
Complete scoped input,Raw inconsistent whitespace/Unicode,Exact expected normalized text,Formatting only; task text preserved
Threshold decision (synthetic),Illustrative .50 cutoff: label 0,Illustrative frozen .20 cutoff: label 1,Shows the actual decision rule; not a measured model improvement
Artifact integrity,Valid frozen copy accepted,Changed copy rejected,Change detected; restored bytes accepted


Model before/after,Reason,Claimed predictive improvement
NOT AVAILABLE in this execution,Private original/V2 validation artifacts and checkpoint are not available,None from this notebook execution


## Gap analysis and improvements

| Observed or unresolved issue | Practical improvement |
|---|---|
| Recorded NLI case is wrong at 99.57% confidence | Use reviewed task-specific labels; assess calibration on adequate data; keep confidence advisory |
| Cleaning and integrity pass, but these do not establish structural reasoning | Review target scope and exhaustivity; evaluate valid renames and role changes separately from ordinary F1 |
| V2 protocol uses provisional function-sensitivity labels | Reconcile labels with declared transitive support membership; do not silently relabel historical results |
| Real checkpoint absent in this environment | Replay the existing verified local V2 checkpoint and inspect the selected failures |
| Simulated failures were recorded correctly | Test operational failures in an authorized run; retain artifacts and investigate rather than auto-retry |

**Fallback:** when real replay cannot run, display `NOT RUN` or `ERROR`. There is no substitute heuristic prediction or invented checkpoint. The deterministic certificate verifier remains independent of semantic advice.

Candidate V3 remains disabled and outside this notebook. Human annotation review and experiment authorization remain separate prerequisites.

## Evidence

Only observations produced in this execution are included below. Hardware/backend simulations and synthetic probability inputs are labeled. No aggregate model accuracy is estimated from these selected examples.

In [12]:
table(evidence, ["Area", "Experiment", "Actual observation"])

Area,Experiment,Actual observation
Historical semantic failure,Read recorded case 003,Neutral annotation; contradiction at 99.57%; no fresh inference
Input handling,Real clean_rows on a synthetic pair,"Whitespace/NFC cleaned; scope, negation and 17 preserved"
Threshold behavior,Real reporter; injected probability 0.30,Threshold .20 yields label 1 with probability .30; NaN rejected
Artifact integrity,Alter disposable validation text,ValueError: Empty or altered split: validation
Execution control,Comparison before training,Rejected; zero attempts; manifest unchanged
Failure handling,Synthetic backend exception + retry,ERROR retained in both records; retry rejected
CPU-only guard,Simulated GPU availability,Rejected and recorded ERROR; training callback not called
Real model inference,Private validation checkpoint replay,NOT RUN: Private V2 bundle not configured; no model outputs generated.


## Conclusion

**Problem:** confidence alone cannot establish a correct dependency decision, and uncontrolled changes can undermine experimental evidence.

**What V2 provides:**
- Complete-input cleaning and split validation.
- Frozen checkpoint/threshold selection and prediction reporting.
- Hash-bound artifact verification.
- Ordered execution and preserved attempt failures.

**Experiments:** the populated cells show what these utilities actually did. The private replay's observed status below determines whether this notebook also demonstrates trained-model behavior.

**Limitations:** synthetic reliability fixtures do not demonstrate model accuracy; historical NLI is a separate task; annotations remain provisional; selected validation examples do not establish generalization or calibration. No V3 approval, training or evaluation is implied.

In [13]:
observed_utilities = sum(e["Area"] != "Real model inference" for e in evidence)
display(Markdown(f"**This execution:** {observed_utilities} evidence entries from historical/utility demonstrations; "
                 f"real checkpoint replay **{replay_status}**. "
                 + ("Inspect the per-example annotation and replay comparisons above."
                    if replay_status == "EXECUTED" else
                    "No fresh trained-model predictions or predictive improvement are claimed.")))
scratch.cleanup()
print("Disposable fixtures removed. Repository source and private run artifacts were not modified.")

Disposable fixtures removed. Repository source and private run artifacts were not modified.


**This execution:** 7 evidence entries from historical/utility demonstrations; real checkpoint replay **NOT RUN**. No fresh trained-model predictions or predictive improvement are claimed.